<picture>
  <source media="(prefers-color-scheme: dark)" srcset="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-dark.svg">
  <img alt="Meep²" src="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-light.svg" width="239" height="80">
</picture>

# 04 · GRPO: rewards a program checks

> **Licence: Apache-2.0.** Copyright Alejandro Martínez Corriá and the Thinkube contributors.
> This notebook, [meepsquared](https://github.com/thinkube/meepsquared) and every package meepsquared installs are permissively licensed:
> you can use, change and ship what you build with them, commercially or not. See [Licensing](https://thinkube.org/meepsquared/licensing.html).

**Where it runs.** On any Linux machine with an NVIDIA GPU of the Ampere generation or later and Meep² installed ([Install](https://github.com/thinkube/meepsquared#install)). The model and the data come from the Hugging Face Hub. Run the cells from top to bottom.

A few cells are marked **Thinkube only**. [Thinkube](https://thinkube.org) is the open-source AI platform Meep² is developed on; on a Thinkube cluster these cells read the model from the cluster's copy, record the run and serve the result. Elsewhere they do nothing, so run every cell as it comes.

The outputs saved in this notebook come from a run on a Thinkube cluster, on an NVIDIA DGX Spark, so they show the cluster's model paths and its run links.

## What you will do

Teach a model a skill by rewarding it, with a training job (`m2.TrainingJob(task="grpo")`), which trains with TRL's `GRPOTrainer`. Notebooks 01 to 03 are supervised: the model imitates answers you give it. Here nobody writes the answers. The model tries, a program grades each try, and the model learns from the grades.

### How GRPO works

**GRPO** (group relative policy optimization) is the reinforcement-learning method behind the reasoning models of 2025. For each prompt the model generates a *group* of answers (8 here). A **reward function** scores each one. Within the group, answers above the group's mean are pushed up and answers below are pushed down: the model learns what distinguishes its better tries from its worse ones, with no separate value model and no human-written target. Most of a step's time is spent generating, which is why GRPO is slow per step and why generation speed matters.

### The task

A **zebra puzzle**: a few houses, a few categories (colour, pet, drink...), and clues like "the owner of the dog lives next to the green house". The model is given the clues and asked to write each one as a formal rule. A constraint solver then deduces the grid from the model's rules, and the reward is the share of the grid's cells the solver gets right. The reward is computed by a program, with no judge model: it is cheap, exact, and cannot be flattered. The task's code is next to the GRPO benchmark, in `benchmarks/grpo/`.

### What to expect

The defaults run 4 steps of 4 prompts × 8 answers of up to 800 tokens, about 10 minutes on one GPU: enough to see the mechanics and the reward move a little. Learning a skill takes a few hundred steps; set `STEPS` and `PUZZLES` for that. With vLLM installed in the kernel's environment, `VLLM = True` generates through vLLM in the training process (the job's `m2.Overrides(vllm=True)`, TRL's colocate mode), several times faster; without vLLM the model generates with Transformers.

Needs `meepsquared`, `pandas` and `python-constraint` (the solver); on Thinkube, the `meepsquared` kernel has them.

## Settings

Every choice of the notebook is a name in this cell, so you can rerun the notebook with other values without touching the cells below, and a notebook job can override any of them with its parameters.

In [2]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "huggingface"; "thinkube" only on a Thinkube cluster
MODEL = PARAMETERS.get("MODEL", "Qwen/Qwen3.5-4B")
STEPS = PARAMETERS.get("STEPS", 4)
PUZZLES = PARAMETERS.get("PUZZLES", 64)                      # distinct puzzles; each step takes 4 of them
NUM_GENERATIONS = PARAMETERS.get("NUM_GENERATIONS", 8)       # answers per prompt: the group GRPO compares
MAX_COMPLETION = PARAMETERS.get("MAX_COMPLETION", 800)       # tokens per answer
MAX_LENGTH = PARAMETERS.get("MAX_LENGTH", 3072)              # tokens of a prompt and its answer: the clues are long
LEARNING_RATE = PARAMETERS.get("LEARNING_RATE", 1e-5)        # ten times smaller than for SFT: RL steps are noisy
RANK = PARAMETERS.get("RANK", 16)
VLLM = PARAMETERS.get("VLLM", False)                         # generate through vLLM in the training process
OUTPUT_DIR = PARAMETERS.get("OUTPUT_DIR", "out/04-grpo-zebra")
# The folder of the zebra task's code: benchmarks/grpo of meepsquared, next to this notebook's folder
TASK_DIR = PARAMETERS.get("TASK_DIR", "../benchmarks/grpo")
# Thinkube only: ignored elsewhere
EXPERIMENT = PARAMETERS.get("EXPERIMENT", "meepsquared")
RUN_NAME = PARAMETERS.get("RUN_NAME", f"04-grpo-{MODEL.split('/')[-1]}")

## Where the model comes from

A fine-tuning starts from a **base model**: a model somebody already trained on a great deal of text, whose behaviour you will adjust with your own data. The base model is the largest thing a training handles, 8 to 70 GB of weights for the models in these notebooks, so where it comes from matters.

`MODEL_SOURCE` chooses:

* `"huggingface"`: the model is downloaded from the Hugging Face Hub by its id, once, into the Hub's cache on this machine. A *gated* model, such as Llama, needs its licence accepted on its Hub page and the token of your account in the `HF_TOKEN` variable.
* `"thinkube"`, only on a [Thinkube](https://thinkube.org) cluster: the model comes from the cluster's **model mirror**: a copy made once into the cluster's shared storage, which every GPU node reads. Nothing is downloaded by a training, a gated model needs no token here, and a 70 GB model loads in about a minute. A model is mirrored in Thinkube Control (AI > Models). `thinkube_models.mirrored_model_path` finds its folder, and stops with an error naming the model if it is not mirrored, before any time is spent.

In [3]:
import os
from pathlib import Path

if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    MODEL_PATH = tkm.mirrored_model_path(MODEL)
    print(f"{MODEL}: {MODEL_PATH}")
elif MODEL_SOURCE == "huggingface":
    MODEL_PATH = MODEL
    print(f"{MODEL}: from the Hugging Face Hub")
else:
    raise ValueError(f"MODEL_SOURCE must be 'thinkube' or 'huggingface', not {MODEL_SOURCE!r}")

Qwen/Qwen3.5-4B: /home/thinkube/thinkube/mlflow/artifacts/1/d2e146295f5e45fd83712013ad80bc42/artifacts/model


## The task

Three functions of `zebra_task` do the work. `make_puzzles` writes puzzles with a fixed seed, so a run is repeatable. `puzzle_to_row` turns one into a prompt for the model and the facts the reward needs (the solution grid, the categories, the size). `zebra_reward_func` parses the model's rules, runs the solver, and grades the result between 0 and 1.

A GRPO dataset has a `prompt` column, a conversation the model continues, and whatever extra columns the reward function needs; the training job keeps them, and TRL passes them to it by name. The cell prints the first prompt, so you can see what the model is asked.

In [4]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(TASK_DIR).resolve()))
import zebra_task
from datasets import Dataset

puzzles = zebra_task.make_puzzles(PUZZLES, seed=1)
train_data = Dataset.from_list([zebra_task.puzzle_to_row(p) for p in puzzles])
print(train_data[0]["prompt"][0]["content"][:1200])

Translate each zebra-puzzle clue into one formal rule.

Categories and values:
  tapa: patatas bravas, croquetas, boquerones, tortilla, pulpo
  wine: rioja, albariño, verdejo, cava, tempranillo
  region: Galicia, Catalonia, Andalusia, Basque Country, La Rioja
  diner: Marta, Joan, Núria, Pep, Laia
  time: 19:00, 20:00, 21:00, 22:00, 23:00

Houses are numbered 1 to 5 from left to right.
'immediate_left' means a is directly left of b; 'left_of' means a is anywhere left of b.
'one_between' means exactly one house stands between a and b.

Rule forms (JSON, one per clue):
{"type":"at","item":["category","value"],"house":2}
{"type":"not_at","item":["category","value"],"house":2}
{"type":"same","a":["category","value"],"b":["category","value"]}
{"type":"immediate_left","a":["category","value"],"b":["category","value"]}
{"type":"immediate_right","a":["category","value"],"b":["category","value"]}
{"type":"next_to","a":["category","value"],"b":["category","value"]}
{"type":"left_of","a":["catego

## Thinkube only: the training run in Thinkube Experiments

*Elsewhere the cell only sets `REPORT_TO` to an empty list; run it anyway, the training job below uses it.*

A training is an experiment: settings in, a loss curve and a model out. Keeping that record is what lets you compare two runs later, or trace a served model back to the data and settings that made it. On Thinkube the record is **Thinkube Experiments** (MLflow). The cell opens a *run* there, before the training job is described: the job's `report_to=REPORT_TO` then sends every step's reward and loss to that run as it trains, so the curves are on the experiments page while the run goes. Elsewhere the metrics are in the trainer's log below, and `REPORT_TO` is empty.

In [5]:
RUN_ID = None
if MODEL_SOURCE == "thinkube":
    import mlflow
    import thinkube_models as tkm

    os.environ["MLFLOW_TRACKING_TOKEN"] = tkm.get_mlflow_token()
    mlflow.set_tracking_uri("http://mlflow.mlflow.svc.cluster.local:5000")
    experiment = mlflow.get_experiment_by_name(EXPERIMENT)
    experiment_id = experiment.experiment_id if experiment else mlflow.create_experiment(EXPERIMENT)
    run = mlflow.start_run(experiment_id=experiment_id, run_name=RUN_NAME)
    RUN_ID = run.info.run_id
    REPORT_TO = ["mlflow"]
    print(f"run {RUN_ID} in experiment {EXPERIMENT!r}: open Thinkube Experiments to follow it")
else:
    REPORT_TO = []

run 0bd1273bbb0543e6a01e48513eb59514 in experiment 'meepsquared': open Thinkube Experiments to follow it


## The training job and its plan

`m2.TrainingJob(task="grpo")` describes a GRPO training: the model, the prompts, the reward functions and how long. The same LoRA as the supervised notebooks goes on the seven projections of every layer. The adapter is what GRPO trains; the base model stays frozen, which is also what makes the training stable at this learning rate.

The recipe (`m2.Recipe`) holds the training choices, and here also how the answers are generated:

* `generations`: the size of the group, 8 answers per prompt.
* `effective_batch`: answers per update of the weights. `4 * NUM_GENERATIONS` makes an update of 4 prompts, each with its group.
* `max_completion_length`: how long an answer may be; longer answers mean more generation time per step. `max_length`, 3072 tokens, holds the prompt and the answer: the puzzle's clues are long.
* `temperature=1.0`, `top_p=1.0`, `top_k=40`: generation must be *varied*, or the group's answers are all the same and there is nothing to compare.
* `chat_template_kwargs={"enable_thinking": False}`: a direct answer from a thinking model; a chat template without the switch ignores it.
* `learning_rate=1e-5`, `max_grad_norm=0.1`, `weight_decay=0.1`: small, careful steps; the reward signal is noisy. The cosine schedule makes large steps first and small ones at the end.

`m2.Overrides(vllm=VLLM)` chooses how the answers are generated: by Transformers with the training model, or by vLLM in the training process.

`job.plan()` loads the model, adds LoRA, measures the memory, and builds TRL's trainer; nothing is trained yet. The plan prints each setting, with TRL's default beside the ones the recipe sets. With vLLM, its `checkpoint` line says how much memory it keeps for vLLM. TRL's default penalty for drifting from the base model, `beta=0`, stays: no reference model is kept in memory, and with a LoRA adapter the drift is bounded anyway. The job keeps Transformers' defaults for Adam's betas.

In [6]:
import meepsquared as m2

job = m2.TrainingJob(
    model=MODEL_PATH, data=train_data, task="grpo", rewards=[zebra_task.zebra_reward_func], output=OUTPUT_DIR,
    steps=STEPS, report_to=REPORT_TO,
    recipe=m2.Recipe(
        rank=RANK, learning_rate=LEARNING_RATE, schedule="cosine", warmup_steps=max(1, STEPS // 10),
        optimizer="paged_adamw_8bit", max_grad_norm=0.1, weight_decay=0.1,
        effective_batch=4 * NUM_GENERATIONS, generations=NUM_GENERATIONS,
        max_length=MAX_LENGTH, max_completion_length=MAX_COMPLETION,
        temperature=1.0, top_p=1.0, top_k=40, chat_template_kwargs={"enable_thinking": False}),
    overrides=m2.Overrides(vllm=VLLM),
)
print(job.plan())  # loads the model, adds LoRA, measures the memory; nothing is trained yet

trainer = job.trainer()  # TRL's GRPOTrainer, set up as job.train() will run it
model, tokenizer = trainer.model, trainer.processing_class

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

╭────────────╮      Meep² 0.1.0: fast LoRA and QLoRA on Transformers, PEFT and TRL
│ meep meep! │      GPU: NVIDIA GB10 (sm_121), allocator: expandable segments. PyTorch 2.11.0a0. Transformers 5.20.2. TRL 1.16.0
╰──┬─────────╯      Model: qwen3_5_text, bfloat16. Liger: on. Padding-free: on
   ╵ ∘ ○  ·  ·  ·   Apache-2.0: https://github.com/thinkube/meepsquared
meepsquared plan for /home/thinkube/thinkube/mlflow/artifacts/1/d2e146295f5e45fd83712013ad80bc42/artifacts/model, LoRA GRPO, on NVIDIA GB10 (94 GiB free)
  loading      shard by shard to GPU 0 (the GPU shares the system's memory)
  weights      bfloat16 (method lora)
  attention    flash_attention_2 (FlashAttention 2 with packed sequences)
  batches      padded (GRPO's batches are TRL's)
  Liger        kernels and fused loss, without its SwiGLU: the fused LoRA MLP runs instead (from_pretrained's default)
  fused LoRA   on (the model's MLP has a fused kernel)
  compile      layers not compiled (FlashAttention runs outside them)
  

/home/thinkube/venvs/custom/meepsquared/arm64/lib/python3.12/site-packages/meepsquared/_trl.py:258: FutureWarning: `use_liger_kernel=True` is deprecated and will be removed in v2.0.0. Use the Hub kernels instead, with `model_init_kwargs={"use_kernels": True}`.
  trainer_init(*bound.args, **bound.kwargs)


## Before training: one attempt, graded

One puzzle, the model's rules, and the reward the solver gives them. `for_inference` and `for_training` around the generation leave the model as the plan set it. A base model usually writes plausible-looking rules in the wrong format, or gets a few right: a reward well below 1. Keep the number; the same puzzle comes back after training.

In [7]:
import torch


def attempt(model, tokenizer, row):
    m2.MeepLanguageModel.for_inference(model)
    try:
        prompt = tokenizer.apply_chat_template(row["prompt"], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_COMPLETION, do_sample=False)
        text = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    finally:
        m2.MeepLanguageModel.for_training(model)
    reward = zebra_task.zebra_reward_func([text], [row["puzzle_solution"]], [row["puzzle_categories"]],
                                          [row["puzzle_n"]], [row["num_clues"]])[0]
    return text, reward


SAMPLE = train_data[0]
text, reward = attempt(model, tokenizer, SAMPLE)
print(f"reward {reward:.2f}\n{text[:800]}")

reward 0.17
```json
[
  {"type":"immediate_right","a":["diner","Galicia"],"b":["tapa","pulpo"]},
  {"type":"next_to","a":["diner","Laia"],"b":["tapa","tortilla"]},
  {"type":"immediate_right","a":["diner","Joan"],"b":["tapa","boquerones"]},
  {"type":"immediate_left","a":["diner","Laia"],"b":["diner","22:00"]},
  {"type":"next_to","a":["diner","22:00"],"b":["diner","Andalusia"]},
  {"type":"not_at","a":["tapa","patatas bravas"],"house":3},
  {"type":"next_to","a":["diner","Basque Country"],"b":["tapa","tortilla"]},
  {"type":"immediate_right","a":["diner","Catalonia"],"b":["wine","albariño"]},
  {"type":"left_of","a":["wine","tempranillo"],"b":["wine","albariño"]},
  {"type":"next_to","a":["diner","Núria"],"b":["diner","Andalusia"]},
  {"type":"two_between","a":["diner","Catalonia"],"b":["wine","verde


## Train

`job.train()` prints the plan again, then trains with TRL's `GRPOTrainer` and every setting of the plan. Each step generates a group of answers for each prompt, grades them with the reward function, and moves the adapter towards the answers that scored above their group's mean. It returns a result: the steps, the time and the peak memory.

The trainer logs the mean reward of each step and its standard deviation within the groups. A standard deviation of zero means every answer got the same grade and the step taught nothing.

In [8]:
result = job.train()
print(f"{result.steps} steps in {result.seconds:.0f} s; peak memory {result.peak_memory_bytes / 2**30:.1f} GiB")

meepsquared plan for /home/thinkube/thinkube/mlflow/artifacts/1/d2e146295f5e45fd83712013ad80bc42/artifacts/model, LoRA GRPO, on NVIDIA GB10 (94 GiB free)
  loading      shard by shard to GPU 0 (the GPU shares the system's memory)
  weights      bfloat16 (method lora)
  attention    flash_attention_2 (FlashAttention 2 with packed sequences)
  batches      padded (GRPO's batches are TRL's)
  Liger        kernels and fused loss, without its SwiGLU: the fused LoRA MLP runs instead (from_pretrained's default)
  fused LoRA   on (the model's MLP has a fused kernel)
  compile      layers not compiled (FlashAttention runs outside them)
  checkpoint   26 of 32 layers (one layer for 32 x 3072 tokens, the bound: GRPO generates its completions as it trains: 11.73 GiB keeping everything, 0.51 recomputed whole; its forward and backward 1816 / 2480 ms in that order; free: 94 of 122 GiB; selective recomputation not offered: the linear-attention layers run flash-linear-attention's delta-rule kernel, whi

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss


4 steps in 544 s; peak memory 24.2 GiB


### Reading the rewards

`reward` is the mean grade of the step's 32 answers; `reward_std` how much they differed within their groups. Over a few hundred steps the mean climbs as the model learns the rule format first, then the harder clue types. Four steps show the first movement, and some noise: in the run saved here the mean reward of the step's answers went 0.29, 0.23, 0.40, 0.42. A 60-step run of the same task and settings on a GB10 (an earlier run, with Unsloth) took the mean from 0.35 over its first 20 steps to 0.49 over its last 20, single steps reaching 0.86; `STEPS = 60` reproduces it, about 2 hours and 20 minutes on a GB10 at 136 s a step.

In [9]:
import pandas as pd

log = pd.DataFrame([h for h in trainer.state.log_history if "reward" in h])
display(log[["step", "reward", "reward_std"] + [c for c in log.columns if c.startswith("rewards/")]].set_index("step").round(3))

,reward,reward_std,rewards/zebra_reward_func/mean,rewards/zebra_reward_func/std
step,,,,
1,0.286,0.362,0.286,0.362
2,0.231,0.275,0.231,0.275
3,0.396,0.337,0.396,0.337
4,0.417,0.340,0.417,0.340


## After training: the same puzzle

The same puzzle, graded again. After four steps the grade of one puzzle is noise: it can move up, stay, or even fall, as the model's one greedy answer to this puzzle changes. The signal of the training is the mean reward over the groups, in the table above. Hundreds of steps teach the format and most of the rules, and then the grade of a single puzzle rises too. In the run saved here it stayed at 0.17, with the mean reward over the groups up from 0.29 to 0.42.

In [10]:
text, reward_after = attempt(model, tokenizer, SAMPLE)
print(f"reward before {reward:.2f}, after {reward_after:.2f}\n{text[:800]}")

reward before 0.17, after 0.17
```json
[
  {"type":"immediate_right","a":["diner","Galicia"],"b":["tapa","pulpo"]},
  {"type":"next_to","a":["diner","Laia"],"b":["tapa","tortilla"]},
  {"type":"immediate_right","a":["diner","Joan"],"b":["tapa","boquerones"]},
  {"type":"immediate_left","a":["diner","Laia"],"b":["diner","22:00"]},
  {"type":"next_to","a":["diner","22:00"],"b":["diner","Andalusia"]},
  {"type":"not_at","a":["tapa","patatas bravas"],"house":3},
  {"type":"next_to","a":["diner","Basque Country"],"b":["tapa","tortilla"]},
  {"type":"immediate_right","a":["diner","Catalonia"],"b":["wine","albariño"]},
  {"type":"left_of","a":["wine","tempranillo"],"b":["wine","albariño"]},
  {"type":"next_to","a":["diner","Núria"],"b":["diner","Andalusia"]},
  {"type":"two_between","a":["diner","Catalonia"],"b":["wine","verde


## Save

The adapter, as in the supervised notebooks. It loads onto the base model with PEFT; notebook 01 shows the merged save and, on Thinkube, the registration for serving.

In [11]:
result.save(f"{OUTPUT_DIR}/adapter")
print(f"adapter saved to {OUTPUT_DIR}/adapter")

if RUN_ID:
    mlflow.end_run()

adapter saved to out/04-grpo-zebra/adapter
🏃 View run 04-grpo-Qwen3.5-4B at: http://mlflow.mlflow.svc.cluster.local:5000/#/experiments/6/runs/0bd1273bbb0543e6a01e48513eb59514
🧪 View experiment at: http://mlflow.mlflow.svc.cluster.local:5000/#/experiments/6


## Where to go next

* [05 · Several GPUs](05-several-gpus.ipynb).
* [GRPO](https://thinkube.org/meepsquared/grpo.html) in the documentation, and the [GRPO benchmark](https://thinkube.org/meepsquared/benchmarks.html#_grpo_on_one_nvidia_gb10).